In [1]:
import pulp
from pulp import *
import numpy as np
import math
import time

In [2]:
PC = [150, 230, 260] # Planting Cost
BP = [238, 210] # Buying Cost
SP = [170, 150, 36, 10] # Saling Price
MR = [200, 240] # minimum Required

kisi = [[2, 2.4, 16],
        [2.5, 3, 20],
        [3, 3.6, 24]]

I = range(3)
w = range(4)
y = range(2)

S1 = range(3)
S2 = range(9)

p = [[1],
    [1/3 for s in S1],
    [1/9 for s in S2]]

scenario_n = 3

start_time = time.time()

In [3]:
scenario_n = 3

NLDS_tk = [[0 for k in range(int(scenario_n ** (t - 1)))]for t in range(1,4)]

e_tk = [[[0] for k in range(int(scenario_n ** (t - 1)))] for t in range(1, 4)]

s_tk = [[0 for k in range(int(scenario_n ** (t - 1)))]for t in range(1,4)]  

pi_tk = [[0 for k in range(int(scenario_n ** (t - 1)))]for t in range(1,4)] 

sigma_tk = [[0 for k in range(int(scenario_n ** (t - 1)))] for t in range(1, 4)]

h_tk = [[0 for k in range(int(scenario_n ** (t - 1)))] for t in range(1, 4)]

T_tk = [[0 for k in range(int(scenario_n ** t))] for t in range(1, 3)]

DIR = "FORWARD"  

counter = 0  
optimalitycut_counter = 0  
iteration = 0 
optimality = False  


In [4]:
Theta_t1 = LpVariable("Theta1")
X_t1 = LpVariable.matrix("X1", I, lowBound=0) 


Theta_t2 = LpVariable.matrix("Theta2", S1, lowBound=0) 
X_t2 = LpVariable.matrix("X2", (I,S1), lowBound=0)  
Y_t1 = LpVariable.matrix("Y1", (y,S1), lowBound=0) 
W_t1 = LpVariable.matrix("W1", (w,S1), lowBound=0) 


Theta_t3 = LpVariable.matrix("Theta3", S2, lowBound=0) 
Y_t2 = LpVariable.matrix("Y2", (y,S2), lowBound=0) 
W_t2 = LpVariable.matrix("W2", (w,S2), lowBound=0) 

for t in range(1,4):
     
    k_t = 3**(t-1) # number of senarios in period 1 is 1 , 2 is 3 and 3 is 9
    for k in range(1,k_t + 1):
        
        NLDS_Problem = LpProblem(f"NLDS(t:{t}_k:{k})", LpMinimize)
            
        if t == 1:
            
            #Objective Function    
            NLDS_Problem += lpSum(PC[i]*X_t1[i] for i in I) + Theta_t1 

            #Constraints
            NLDS_Problem += lpSum(X_t1[i] for i in I) <= 500 

            NLDS_Problem += Theta_t1 == 0, "theta=0"
            
            NLDS_Problem.solve()


        if t == 2:

            
            NLDS_Problem += (lpSum(PC[i]*X_t2[i][k-1] for i in I) \
                             + (lpSum(BP[i]*Y_t1[i][k-1] for i in y) \
                                      - lpSum(SP[i]*W_t1[i][k-1] for i in w)) \
                             + Theta_t2[k-1]) 

            
            NLDS_Problem += lpSum(X_t2[i][k-1] for i in I) <= 500 

            for i in y:
                NLDS_Problem += kisi[k-1][i]*(X_t1[i].varValue) + Y_t1[i][k-1] - W_t1[i][k-1] >= MR[i] 

            NLDS_Problem += kisi[k-1][i]*(X_t1[2].varValue) - W_t1[2][k-1] - W_t1[3][k-1] >= 0 
            NLDS_Problem += W_t1[2][k-1] <= 6000 

            NLDS_Problem += (X_t1[2].varValue) + X_t2[2][k-1] <= 500

            NLDS_Problem += Theta_t2[k-1] == 0, "theta=0"
            
            NLDS_Problem.solve()  


        if t == 3:        
            a_k = (k-1)//3 # determine parent of happened senario of k in period 3
            kisi3 = (k-1)%3 + 1 # determine which kisi is happening in period 3

            #Objective Function
            NLDS_Problem += (lpSum(BP[i]*Y_t2[i][k-1] for i in y) \
                                   - lpSum(SP[i]*W_t2[i][k-1] for i in w)) + Theta_t3[k-1]

            #Constraints
            for i in y:
                NLDS_Problem += kisi[kisi3-1][i]*(X_t2[i][a_k].varValue) + Y_t2[i][k-1] - W_t2[i][k-1] >= MR[i]

            NLDS_Problem += kisi[kisi3-1][i]*(X_t2[2][a_k].varValue) - W_t2[2][k-1] - W_t2[3][k-1] >= 0 
            NLDS_Problem += W_t2[2][k-1] <= 6000

            NLDS_Problem += Theta_t3[k-1] == 0, "theta=0"
                      
        NLDS_tk[t-1][k-1] = NLDS_Problem

In [5]:
def Calculate_Matrices(t,k): 

    global h_tk, T_tk
    
    if t == 1:
        h = [500]
        
  
    if t == 2:
        h = [-500, MR[0], MR[1], 0, -6000, -500]
        
        T = [[0 for v in range(3)] for m in range(6)]
        
        
        for m in range(1,4):
            v = m-1
            T[m][v] = kisi[k-1][v]
            
        T[-1][-1] = -1

    
    if t == 3:
        h = [MR[0], MR[1], 0, -6000]
        
        T = [[0 for v in range(3)] for m in range(4)]
        
        kisi3 = (k-1)%3 + 1 # determine which kisi is happening in period 3
        for m in range(3):
            v = m
            T[m][v] = kisi[kisi3-1][v]

    
    h_tk[t-1][k-1] = h
    if t != 1:
        T_tk[t-2][k-1] = T

In [6]:
for t in range(1,4):
    k_t = 3**(t-1)
    for k in range(1,k_t + 1):
        Calculate_Matrices(t,k)

In [7]:
def Check_Feasibility(NLDS):
    
    problem_status = LpStatus[NLDS.status]
    if problem_status == -1 or problem_status == -2:   # -1 means infeasible and -2 means infinity
        if t == 1:
            print("The problem is Infeasible!")
        else:
            
            DIR = "BACK"
            t -= 1
            k = (k-1)//3
            Step_1(t,k)

In [8]:
def Update_Next_Stage_NLDS(t,k):
    
    global NLDS_tk
    
    u  = t + 1
    k_u = 3**(u-1)
    for k in range(1,k_u + 1):
        
        if u == 2:
            for c in range(2, 4):
                i = c-2
                NLDS_tk[u-1][k-1].constraints[f"_C{c}"] = (kisi[k-1][i]*(X_t1[i].varValue) \
                                                             + Y_t1[i][k-1] - W_t1[i][k-1] >= MR[i])
            
            
            NLDS_tk[u-1][k-1].constraints["_C4"] = (kisi[k-1][i]*(X_t1[3-1].varValue) \
                                                      - W_t1[3-1][k-1] - W_t1[4-1][k-1] >= 0)
            
            NLDS_tk[u-1][k-1].constraints["_C6"] = (X_t1[3-1].varValue) + X_t2[3-1][k-1] <= 500
            
            
        if u == 3:
            kisi3 = (k-1)%3 + 1
            for c in range(1,3):
                i = c-1
                NLDS_tk[u-1][k-1].constraints[f"_C{c}"] = (kisi[kisi3-1][i]*(X_t2[i][a_k].varValue) \
                                                             + Y_t2[i][k-1] - W_t2[i][k-1] >= MR[i])
                
            NLDS_tk[u-1][k-1].constraints["_C3"] = (kisi[kisi3-1][i]*(X_t2[3-1][a_k].varValue) \
                                                      - W_t2[3-1][k-1] - W_t2[4-1][k-1] >= 0)

In [9]:
def Dual_Values(NLDS,t,k):
    
    # to add Dual value of each constraint to its related variable
    
    global pi_tk, sigma_tk
    
    pi = list()
    sigma = list()
    for name, c in list(NLDS.constraints.items()):
        if name != "theta=0" and name != "feasibility_cut" and name[:-1] != "optimality_cut":
            pi.append(c.pi)
            
        if name[:-1] == "optimality_cut":
            sigma.append(c.pi)
    
    pi_tk[t-1][k-1] = pi
    sigma_tk[t-1][k-1] = sigma

In [10]:
def Step_1(t,k):
    
    if optimality == True:
        return
    
    global NLDS_tk
    global DIR
    
    NLDS = NLDS_tk[t-1][k-1]
    NLDS.solve()

    print(f"\n NLDS(t:{t}-k:{k})")
    print("Objective Value:", NLDS.objective.value())
           
    Check_Feasibility(NLDS)
    
    if t != 3:
        Update_Next_Stage_NLDS(t,k)
      
    Dual_Values(NLDS,t,k)
    
    k_t = 3**(t-1)
    if k < k_t:
        k += 1
        Step_1(t,k)
        
    else:
        if t == 1:
            DIR = "FORE"

        if t < 3 and DIR == "FORE":
            t += 1
            k = 1
            
            Step_1(t,k)
            
        elif t == 3:
            DIR = "BACK"
    
        if DIR == "BACK":       
            Step_2(t,k)

In [11]:
def Matrix_E(u,j):

    E = list()
    
    C_k = range(3*(j-1)+1,3*j+1)  
       
    pi_T_uk = [np.dot(np.array(pi_tk[u][k-1]),np.array(T_tk[u-1][k-1])).tolist() for k in C_k]
    
    for v in range(len(pi_T_uk[0])):
        E_v = sum((p[u][k]/p[u-1][j-1])*pi_T_uk[k][v] for k in range(len(pi_T_uk)))
        E.append(E_v)
    
    return E

In [12]:
def Matrix_e(u,j):
    
    global e_tk, s_tk
        
    C_k = range(3*(j-1),3*j)
    
    pi_h_uk = [np.dot(np.array(pi_tk[u][k-1]),np.array(h_tk[u][k-1])) for k in C_k]
    sigma_e_tk = [sum(sigma_tk[u][k][m]*e_tk[u][k][m+1] for m in range(s_tk[u][k])) for k in C_k]
    
    e = sum((p[u][k-1]/p[u-1][j-1])*(pi_h_uk[k] + sigma_e_tk[k]) for k in range(len(pi_h_uk)))
    
    e_tk[u-1][j-1].append(e)
    
    return e

In [13]:
def Optimality_Cut(t,k,E,e):
    
    global NLDS_tk
    
    if t == 1:        
        NLDS_tk[t-1][k-1] += lpSum(E[i]*X_t1[i] for i in range(len(E))) + Theta_t1 >= e, f"optimality_cut{counter}"
        
    if t == 2:
        NLDS_tk[t-1][k-1] += lpSum(E[i]*X_t2[i][k-1] for i in range(len(E))) + Theta_t2[k-1] >= e, f"optimality_cut{counter}"

In [14]:
def Step_2(t,k):
    
    global optimality
    
    if optimality == True:
        return

    
    global NLDS_tk, counter,  optimalitycut_counter,  iteration 
      
    optimalitycut_counter = 0

    if t == 1:
        t += 1
        
        iteration += 1
        Step_1(t,k)
        
    else:
        u = t - 1
        k_u = 3**(u-1)
        
        for j in range(1, k_u + 1):
            
            E = Matrix_E(u,j)
            e = Matrix_e(u,j) 

            if u == 1:
                theta_bar = e - sum(E[v]*(X_t1[v].varValue) for v in range(len(E)))
            if u == 2:
                theta_bar = e - sum(E[v]*(X_t2[v][j-1].varValue) for v in range(len(E)))
                
            
            theta_exists = 0
            for name, c in list(NLDS_tk[u-1][j-1].constraints.items()):
                if name == "theta=0":
                    theta_exists = 1
                    del NLDS_tk[u-1][j-1].constraints["theta=0"]
                    Optimality_Cut(u,j,E,e)
                    if u == 1:
                        optimalitycut_counter += 1
                    s_tk[u-1][j-1] = 1
                    counter += 1

            if theta_exists == 0: # constraint "theta=0" does not exist
                if ((u == 1 and theta_bar >  Theta_t1.varValue)
                    or (u == 2 and (theta_bar > Theta_t2[j-1].varValue ))):
                    Optimality_Cut(u,j,E,e)
                    if u == 1:
                        optimalitycut_counter += 1
                    s_tk[u-1][j-1] += 1
                    counter += 1
           
    if t == 2 and optimalitycut_counter == 0:
        
        optimality = True
        return
                
    else:
        t -= 1
        k = 1
                
        if t == 1:
            DIR = "FORE"

    iteration += 1
    Step_1(t,k)

In [15]:
t = 1
k = 1
iteration += 1
Step_1(t,k)


 NLDS(t:1-k:1)
Objective Value: 0.0

 NLDS(t:2-k:1)
Objective Value: 98000.0

 NLDS(t:2-k:2)
Objective Value: 98000.0

 NLDS(t:2-k:3)
Objective Value: 98000.0

 NLDS(t:3-k:1)
Objective Value: 98000.0

 NLDS(t:3-k:2)
Objective Value: 98000.0

 NLDS(t:3-k:3)
Objective Value: 98000.0

 NLDS(t:3-k:4)
Objective Value: 98000.0

 NLDS(t:3-k:5)
Objective Value: 98000.0

 NLDS(t:3-k:6)
Objective Value: 98000.0

 NLDS(t:3-k:7)
Objective Value: 98000.0

 NLDS(t:3-k:8)
Objective Value: 98000.0

 NLDS(t:3-k:9)
Objective Value: 98000.0

 NLDS(t:2-k:1)
Objective Value: 122705.882

 NLDS(t:2-k:2)
Objective Value: 122705.882

 NLDS(t:2-k:3)
Objective Value: 122705.882

 NLDS(t:1-k:1)
Objective Value: -107294.12

 NLDS(t:2-k:1)
Objective Value: 79505.882

 NLDS(t:2-k:2)
Objective Value: 68705.882

 NLDS(t:2-k:3)
Objective Value: 57905.882

 NLDS(t:3-k:1)
Objective Value: 28400.0008

 NLDS(t:3-k:2)
Objective Value: 14400.000999999997

 NLDS(t:3-k:3)
Objective Value: 400.0011999999988

 NLDS(t:3-k:4)
Obj

In [16]:
Objective_Value = 0 

for t in range(1,4):
    k_t = 3**(t-1)
    for k in range(1,k_t + 1):
        
        Objective_Value += p[t-1][k-1]*NLDS_tk[t-1][k-1].objective.value()
        
Objective_Value = round(Objective_Value , 2)


In [17]:
print('Objective Value is' , Objective_Value)

Objective Value is -28423.53


In [18]:
print("run time is: %s seconds " % (round(time.time() - start_time, 2)))

run time is: 2.07 seconds 
